In [ ]:
import queue
import sounddevice as sd
import numpy as np
import librosa
import joblib
from tensorflow.keras.models import load_model
from collections import deque

# ───────── CONFIG ─────────
MODEL_PATH = 'D:\FYP\FYP_AI_lab\husnain_ai_lab_fyp\AI_Code Files\CNN Resnet Models\CNN_SE-Resnet_Optimized.ipynb'
LABEL_ENCODER_PATH = 'label_encoder.pkl'

DEVICE_INDEX = 12        # UMIK-1 fixed device index
UMIK_SR = 48000          # Mic sampling rate
TARGET_SR = 16000        # Model training rate
DURATION = 1.0           # Window length (seconds)
STEP = 0.5               # Time between predictions (seconds)

N_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 512
MAX_MFCC_LENGTH = 40     # Pad/truncate MFCC frames

SMOOTHING_WINDOW = 5     # Rolling average over last N predictions
ALERT_THRESHOLD = 0.6    # Alert threshold for DroneA detection
# ─────────────────────────

# Load model and label encoder
model = load_model(MODEL_PATH)
label_encoder = joblib.load(LABEL_ENCODER_PATH)
classes = label_encoder.classes_
drone_idx = list(classes).index('droneA')

# Buffers
audio_queue = queue.Queue()
buffer = deque(maxlen=int(UMIK_SR * DURATION))
p_history = deque(maxlen=SMOOTHING_WINDOW)
alert_triggered = False

# MFCC extraction
def extract_mfcc_window(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    if mfcc.shape[1] < MAX_MFCC_LENGTH:
        pad = MAX_MFCC_LENGTH - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad)), mode='constant')
    else:
        mfcc = mfcc[:, :MAX_MFCC_LENGTH]
    return mfcc

# Audio input callback
def audio_callback(indata, frames, time, status):
    if status:
        print("⚠️", status)
    audio_queue.put(indata[:, 0].copy())

# Main loop
def main():
    print(f"\n🎧 Starting stream on UMIK-1 (device index {DEVICE_INDEX})...")
    with sd.InputStream(
        samplerate=UMIK_SR,
        device=DEVICE_INDEX,
        channels=1,
        blocksize=int(UMIK_SR * STEP),
        callback=audio_callback
    ):
        print("🛸 Listening for DroneA... (press Ctrl+C to stop)")
        try:
            while True:
                block = audio_queue.get()
                buffer.extend(block)

                if len(buffer) < UMIK_SR * DURATION:
                    continue

                y48 = np.array(buffer)
                y16 = librosa.resample(y48, orig_sr=UMIK_SR, target_sr=TARGET_SR)
                mfcc = extract_mfcc_window(y16, TARGET_SR)
                X = mfcc[np.newaxis, :, :, np.newaxis]

                probs = model.predict(X, verbose=0)[0]
                p_drone = probs[drone_idx]
                p_history.append(p_drone)

                avg_p = np.mean(p_history)

                if avg_p > ALERT_THRESHOLD and not alert_triggered:
                    print(f"\n🚨 ALERT: DroneA detected! Avg Probability: {avg_p:.3f}")
                    alert_triggered = True
                elif avg_p <= ALERT_THRESHOLD:
                    alert_triggered = False

                print(f"DroneA probability (avg): {avg_p:.3f}", end='\r')

        except KeyboardInterrupt:
            print("\n🛑 Stream stopped.")

if __name__ == '__main__':
    main()


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'cnn_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import os
import time
import queue
import threading
import numpy as np
import sounddevice as sd
import librosa
import tensorflow as tf
# ======================
#     CONFIG
# ======================
MODEL_PATH       = r"D:/FYP/FYP_AI_lab/husnain_ai_lab_fyp\AI_Code Files/CNN Resnet Models/CNN_SE-Resnet_Optimized.ipynb"  # <-- update
THRESHOLD        = 0.50
DISPLAY_EVERY_N  = 1           # print every N chunks
CHUNK_SEC        = 1.0
TARGET_SR        = 16000       # enforced
N_MFCC           = 13
TARGET_FRAMES    = 40          # enforce 40 frames for the MFCC time axis
N_FFT            = 1024
HOP_LENGTH       = 400         # ~25ms @16k; we’ll pad/trim to 40 anyway
WIN_LENGTH       = None        # default (n_fft)
CENTER           = True
DEVICE_NAME_HINT = "UMIK"      # set to "" to use default, e.g. "UMIK-1" or partial name

# Optional: smoother readout (exponential moving average over last few probs)
USE_EMA     = True
EMA_DECAY   = 0.8  # higher = smoother (0..1)

# ======================
#  UTILITIES
# ======================
def pick_input_device(name_hint: str | None = None):
    """Pick an input device by (partial) name match, else default input."""
    if not name_hint:
        return None  # default device

    try:
        devices = sd.query_devices()
        for i, d in enumerate(devices):
            if d.get("max_input_channels", 0) > 0 and name_hint.lower() in d["name"].lower():
                print(f"[INFO] Selected input device {i}: {d['name']}")
                return i
        print(f"[WARN] No input device matched '{name_hint}'. Using default.")
        return None
    except Exception as e:
        print(f"[WARN] Could not enumerate devices: {e}. Using default.")
        return None

def enforce_16k(audio: np.ndarray, sr_in: int) -> np.ndarray:
    """Ensure mono 16 kHz. Resample if needed, and flatten to 1-D float32."""
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)  # mono
    if sr_in != TARGET_SR:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr_in, target_sr=TARGET_SR)
    return audio.astype(np.float32)

def mfcc_13x40x1(y_1s_16k: np.ndarray) -> np.ndarray:
    """Compute MFCC and force shape (13, 40, 1)."""
    # Compute MFCC (frames may not equal TARGET_FRAMES due to hop/center)
    mfcc = librosa.feature.mfcc(
        y=y_1s_16k,
        sr=TARGET_SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        center=CENTER
    )  # shape: (13, T)

    # Pad or trim time axis to TARGET_FRAMES
    T = mfcc.shape[1]
    if T < TARGET_FRAMES:
        pad = TARGET_FRAMES - T
        mfcc = np.pad(mfcc, ((0, 0), (0, pad)), mode="constant")
    else:
        mfcc = mfcc[:, :TARGET_FRAMES]

    # Add channel dim -> (13, 40, 1)
    return mfcc[..., np.newaxis].astype(np.float32)

def print_header():
    print("="*70)
    print(" Real-time Drone Detection Demo")
    print(f" Model        : {os.path.basename(MODEL_PATH)}")
    print(f" Threshold    : {THRESHOLD:.2f}")
    print(f" Chunk        : {CHUNK_SEC:.2f}s  | Target SR: {TARGET_SR} Hz")
    print(f" MFCC shape   : (13, {TARGET_FRAMES}, 1)")
    print("="*70)

# ======================
#  MAIN LOOP
# ======================
def main():
    # Load model
    model = tf.keras.models.load_model(MODEL_PATH)
    input_shape = model.inputs[0].shape
    if tuple(input_shape[-3:]) != (N_MFCC, TARGET_FRAMES, 1):
        print(f"[WARN] Model expects input shape {tuple(input_shape[-3:])}, "
              f"but we’re producing {(N_MFCC, TARGET_FRAMES, 1)}.")

    # Choose device (optional by name)
    device_id = pick_input_device(DEVICE_NAME_HINT)

    # Try to open stream *at* TARGET_SR to avoid resampling cost if supported
    desired_sr = TARGET_SR
    chunk_samples = int(round(CHUNK_SEC * desired_sr))

    print_header()
    print("[INFO] Press Ctrl+C to stop.\n")

    # A tiny ring buffer to collect exactly 1s each iteration (blocking read is fine)
    ema_prob = None
    idx = 0

    try:
        with sd.InputStream(device=device_id,
                            samplerate=desired_sr,
                            channels=1,
                            dtype='float32',
                            blocksize=chunk_samples):
            # Warm-up small sleep (some devices need a tick)
            time.sleep(0.1)

            while True:
                # Read exactly 1s of audio
                audio = sd.rec(frames=chunk_samples, samplerate=desired_sr, channels=1, dtype='float32')
                sd.wait()  # blocking until recorded

                y = audio[:, 0]  # mono
                # Enforce 16 kHz anyway (if device coerced to a different rate)
                y = enforce_16k(y, sr_in=desired_sr)

                # If drift produced just-under 1s (rare), pad/crop
                if len(y) < TARGET_SR:
                    y = np.pad(y, (0, TARGET_SR - len(y)))
                else:
                    y = y[:TARGET_SR]

                x = mfcc_13x40x1(y)              # (13,40,1)
                x = x[np.newaxis, ...]           # (1,13,40,1)

                # Predict p(drone)
                probs = model.predict(x, verbose=0)[0]
                p_drone = float(probs[1]) if probs.shape[-1] == 2 else float(probs.squeeze())

                # EMA smoothing (optional)
                if USE_EMA:
                    ema_prob = p_drone if ema_prob is None else (EMA_DECAY*ema_prob + (1-EMA_DECAY)*p_drone)
                    disp_prob = ema_prob
                else:
                    disp_prob = p_drone

                # Print
                alert = "  <<< ALERT >>>" if disp_prob >= THRESHOLD else ""
                if idx % DISPLAY_EVERY_N == 0:
                    print(f"[{idx:05d}] P(drone)={disp_prob:0.3f}{alert}")
                idx += 1

    except KeyboardInterrupt:
        print("\n[INFO] Stopped by user.")
    except Exception as e:
        print(f"[ERROR] {e}")
        print("[HINT] If you get device errors, set DEVICE_NAME_HINT='' to use default input device.")

if __name__ == "__main__":
    main()


In [4]:
import queue
import sounddevice as sd
import numpy as np
import librosa
from tensorflow.keras.models import load_model
from collections import deque
import time

# ───────── CONFIG ─────────
MODEL_PATH = r"D:/FYP/FYP_AI_lab/husnain_ai_lab_fyp\AI_Code Files/Testing Models\drone_resnet_final_form.keras"  # <-- set to your .keras
DEVICE_INDEX = 12        # UMIK-1 fixed device index (change if needed)
UMIK_SR = 48000          # Mic sampling rate
TARGET_SR = 16000        # Model training rate
DURATION = 1.0           # Window length (seconds)
STEP = 0.5               # Time between predictions (seconds) -> 0.5 s hop => 50% overlap

N_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 512
MAX_MFCC_LENGTH = 40     # Pad/truncate MFCC frames (time axis)

SMOOTHING_WINDOW = 5     # Rolling avg over last N probabilities
ALERT_THRESHOLD = 0.6    # Alert threshold for Drone detection
# ─────────────────────────

# Load model (binary softmax: [p(no-drone), p(drone)])
model = load_model(MODEL_PATH)

# Buffers
audio_queue = queue.Queue()
buffer = deque(maxlen=int(UMIK_SR * DURATION))
p_history = deque(maxlen=SMOOTHING_WINDOW)
alert_triggered = False

def extract_mfcc_window(y, sr):
    """
    Compute MFCCs and force shape (13, 40).
    Using n_fft=2048, hop_length=512 as requested.
    """
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, center=True
    )  # (13, T)
    T = mfcc.shape[1]
    if T < MAX_MFCC_LENGTH:
        pad = MAX_MFCC_LENGTH - T
        mfcc = np.pad(mfcc, ((0, 0), (0, pad)), mode="constant")
    else:
        mfcc = mfcc[:, :MAX_MFCC_LENGTH]
    return mfcc

def audio_callback(indata, frames, time_info, status):
    if status:
        print("⚠️", status)
    # mono channel 0
    audio_queue.put(indata[:, 0].copy())

def main():
    print(f"\n🎧 Starting stream on device index {DEVICE_INDEX} @ {UMIK_SR} Hz ...")
    # Use blocksize = STEP seconds so we get 0.5s chunks into the ring buffer
    blocksize = int(UMIK_SR * STEP)

    # Warm-up a dummy inference so the first live prediction is fast
    _ = model.predict(np.zeros((1, N_MFCC, MAX_MFCC_LENGTH, 1), dtype=np.float32), verbose=0)

    with sd.InputStream(
        samplerate=UMIK_SR,
        device=DEVICE_INDEX,
        channels=1,
        blocksize=blocksize,
        dtype='float32',
        callback=audio_callback
    ):
        print("🛸 Listening for drone... (press Ctrl+C to stop)")
        try:
            tick = 0
            while True:
                # pull next 0.5 s block
                block = audio_queue.get()
                buffer.extend(block)

                # wait until we have 1.0 s in the buffer
                if len(buffer) < int(UMIK_SR * DURATION):
                    continue

                # assemble 1.0 s window @ 48 kHz
                y48 = np.array(buffer, dtype=np.float32)
                # resample to 16 kHz for the model
                y16 = librosa.resample(y48, orig_sr=UMIK_SR, target_sr=TARGET_SR)

                # (safety) ensure exactly 1.0 s @ 16 kHz
                if len(y16) < TARGET_SR:
                    y16 = np.pad(y16, (0, TARGET_SR - len(y16)))
                else:
                    y16 = y16[:TARGET_SR]

                # MFCC -> (13,40)
                mfcc = extract_mfcc_window(y16, TARGET_SR)
                # add channel dim -> (13,40,1), and batch -> (1,13,40,1)
                X = mfcc[np.newaxis, :, :, np.newaxis].astype(np.float32)

                # inference
                probs = model.predict(X, verbose=0)[0]
                # binary softmax: [no_drone, drone]
                p_drone = float(probs[1])
                p_history.append(p_drone)
                avg_p = float(np.mean(p_history))

                # alert logic
                if avg_p > ALERT_THRESHOLD and not alert_triggered:
                    print(f"\n🚨 ALERT: Drone detected! Avg Probability: {avg_p:.3f}")
                    alert_triggered = True
                elif avg_p <= ALERT_THRESHOLD:
                    alert_triggered = False

                # periodic print
                if tick % 1 == 0:
                    print(f"P(drone) avg over last {SMOOTHING_WINDOW}: {avg_p:.3f}", end="\r")
                tick += 1

        except KeyboardInterrupt:
            print("\n🛑 Stream stopped.")
        except Exception as e:
            print(f"\n[ERROR] {e}")
            print("➡ Try a different DEVICE_INDEX or check microphone permissions.")

if __name__ == '__main__':
    main()



🎧 Starting stream on device index 12 @ 48000 Hz ...
🛸 Listening for drone... (press Ctrl+C to stop)

🛑 Stream stopped.


In [9]:
import sounddevice as sd
for i, d in enumerate(sd.query_devices()):
    print(i, d['name'], "in:", d['max_input_channels'])


0 Microsoft Sound Mapper - Input in: 2
1 Microphone Array (Intel® Smart  in: 3
2 Microsoft Sound Mapper - Output in: 0
3 Speakers (Realtek(R) Audio) in: 0
4 Primary Sound Capture Driver in: 2
5 Microphone Array (Intel® Smart Sound Technology for Digital Microphones) in: 3
6 Primary Sound Driver in: 0
7 Speakers (Realtek(R) Audio) in: 0
8 Speakers (Realtek(R) Audio) in: 0
9 Microphone Array (Intel® Smart Sound Technology for Digital Microphones) in: 1
10 Output (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(DESKTOP-LL7HMU0)) in: 0
11 Input (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(DESKTOP-LL7HMU0)) in: 1
12 Microphone Array 1 () in: 1
13 Microphone Array 2 () in: 1
14 Microphone Array 3 () in: 3
15 Microphone Array 4 () in: 3
16 Speakers () in: 0
17 Headset (@System32\drivers\bthhfenum.sys,#2;%1 Hands-Free%0
;(LED-907)) in: 0
18 Headset (@System32\drivers\bthhfenum.sys,#2;%1 Hands-Free%0
;(LED-907)) in: 1
19 Headphones () in: 0
20 Microphone (Realtek

In [11]:
import sounddevice as sd
import sys

def list_all():
    print("\n=== Host APIs ===")
    try:
        has = sd.query_hostapis()
        for i, ha in enumerate(has):
            print(f"[{i}] {ha['name']}  (default_input_device={ha.get('default_input_device')})")
    except Exception as e:
        print("Failed to query hostapis:", e)

    print("\n=== Devices (all) ===")
    try:
        devs = sd.query_devices()
        for i, d in enumerate(devs):
            print(f"[{i:02d}] in:{d['max_input_channels']:>2} out:{d['max_output_channels']:>2}  {d['name']}  (hostapi={d['hostapi']})")
    except Exception as e:
        print("Failed to query devices:", e)

def find_umiki_like():
    print("\n=== Search for 'umik' (case-insensitive) ===")
    devs = sd.query_devices()
    matches = []
    for i, d in enumerate(devs):
        if d.get('max_input_channels', 0) > 0 and "umik" in d['name'].lower():
            matches.append((i, d))
            print(f"-> [{i}] {d['name']}  (hostapi={d['hostapi']})  in:{d['max_input_channels']}")
    if not matches:
        print("No device with 'umik' found.")
    return matches

def try_open(device_index=None, samplerate=48000):
    print(f"\n=== Try opening device index {device_index} @ {samplerate} Hz ===")
    try:
        with sd.InputStream(device=device_index, channels=1, samplerate=samplerate, dtype='float32'):
            print("SUCCESS: Stream opened.")
    except Exception as e:
        print("FAILED:", e)

list_all()
matches = find_umiki_like()

# Optional: force a specific host API (sometimes helps on Windows)
# Common hostapis: 0=MME, 1=Windows DirectSound, 2=WASAPI, 3=WDM-KS (depends on machine)
# Try WASAPI first:
try:
    sd.default.hostapi = 2  # set to WASAPI if available (adjust after checking hostapi list)
    print("\nSet default hostapi = 2 (WASAPI) — adjust if needed.")
except Exception as e:
    print("Could not set default hostapi:", e)

# If you saw a promising device index from the printed list, test it here:
# Replace 12 with the index you want to test:
try_open(device_index=12, samplerate=48000)



=== Host APIs ===
[0] MME  (default_input_device=1)
[1] Windows DirectSound  (default_input_device=4)
[2] Windows WASAPI  (default_input_device=9)
[3] Windows WDM-KS  (default_input_device=12)

=== Devices (all) ===
[00] in: 2 out: 0  Microsoft Sound Mapper - Input  (hostapi=0)
[01] in: 3 out: 0  Microphone Array (Intel® Smart   (hostapi=0)
[02] in: 0 out: 2  Microsoft Sound Mapper - Output  (hostapi=0)
[03] in: 0 out: 2  Speakers (Realtek(R) Audio)  (hostapi=0)
[04] in: 2 out: 0  Primary Sound Capture Driver  (hostapi=1)
[05] in: 3 out: 0  Microphone Array (Intel® Smart Sound Technology for Digital Microphones)  (hostapi=1)
[06] in: 0 out: 2  Primary Sound Driver  (hostapi=1)
[07] in: 0 out: 2  Speakers (Realtek(R) Audio)  (hostapi=1)
[08] in: 0 out: 2  Speakers (Realtek(R) Audio)  (hostapi=2)
[09] in: 1 out: 0  Microphone Array (Intel® Smart Sound Technology for Digital Microphones)  (hostapi=2)
[10] in: 0 out: 1  Output (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;

In [16]:
import pyaudio
p = pyaudio.PyAudio()
for i in range(p.get_device_count()):
    info = p.get_device_info_by_index(i)
    print(i, info.get('name'), "in:", info.get('maxInputChannels'))
p.terminate()


0 Microsoft Sound Mapper - Input in: 2
1 Microphone (Umik-1  Gain: 18dB) in: 2
2 Microphone Array (IntelÂ® Smart  in: 3
3 Microsoft Sound Mapper - Output in: 0
4 Speakers (Realtek(R) Audio) in: 0
5 Primary Sound Capture Driver in: 2
6 Microphone (Umik-1  Gain: 18dB) in: 2
7 Microphone Array (IntelÂ® Smart Sound Technology for Digital Microphones) in: 3
8 Primary Sound Driver in: 0
9 Speakers (Realtek(R) Audio) in: 0
10 Speakers (Realtek(R) Audio) in: 0
11 Microphone Array (IntelÂ® Smart Sound Technology for Digital Microphones) in: 1
12 Microphone (Umik-1  Gain: 18dB) in: 2
13 Output (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(DESKTOP-LL7HMU0)) in: 0
14 Input (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(DESKTOP-LL7HMU0)) in: 1
15 Microphone Array 1 () in: 1
16 Microphone Array 2 () in: 1
17 Microphone Array 3 () in: 3
18 Microphone Array 4 () in: 3
19 Speakers () in: 0
20 Headset (@System32\drivers\bthhfenum.sys,#2;%1 Hands-Free%0
;(LED-907)) in: 0
